In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"


In [2]:
os.environ["HOME"] = "/mnt/nas/shuvranshu"
os.environ["HF_HOME"] = "/mnt/nas/shuvranshu/huggingface_cache"
os.environ["TRANSFORMERS_CACHE"] = "/mnt/nas/shuvranshu/huggingface_cache"
os.environ["HF_DATASETS_CACHE"] = "/mnt/nas/shuvranshu/huggingface_cache"
os.environ["XDG_CACHE_HOME"] = "/mnt/nas/shuvranshu/huggingface_cache"
os.environ["HF_DATASETS_CACHE"] = "/mnt/nas/shuvranshu/huggingface_cache"
os.makedirs("/mnt/nas/shuvranshu/huggingface_cache", exist_ok=True)

In [3]:
from langchain_community.document_loaders import   DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
import faiss
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
import numpy as np
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
#hf token 
load_dotenv()  
hf_token = os.getenv("HF_TOKEN")
os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token

/mnt/nas/shuvranshu/.conda/envs/ragenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from datasets import load_dataset

# Load squadv2 (use 'distractor' for RAG-style evaluation)
# dataset = load_dataset("squad_v2", split="validation")
dataset=load_dataset("rajpurkar/squad_v2",split="validation[:100]")

# small subset for testing
# dataset = dataset.select(range(100))  # start small

In [5]:
dataset[4]["answers"]["text"]
# len(dataset)

['10th century', 'the first half of the 10th century', '10th', '10th']

In [ ]:
# ground_truths=[]

# for item in dataset:
#     if(len(item["answers"]["text"])):
#         ground_truths.append(item["answers"]["text"][0])
#     else:
#         ground_truths.append("")

In [ ]:
# from collections import Counter

# def get_most_frequent_answer(row):
#     ans_list = row['answers']['text']
    
#     # Check if the list is empty (unanswerable questions)
#     if not ans_list:
#         return "N/A"
    
#     # Count occurrences of each unique string
#     counts = Counter(ans_list)
    
#     # .most_common(1) returns a list like [('France', 4)]
#     # We take the first element [0] and then the text [0]
#     most_common_answer = counts.most_common(1)[0][0]
    
#     return most_common_answer

# # Example usage:
# # row = {'answers': {'text': ['France', 'France', 'France', 'France']}}
# # result -> 'France'

In [ ]:
# loader= DirectoryLoader('documents',glob='*.pdf',loader_cls=PyPDFLoader)
# docs=loader.lazy_load()

In [ ]:
# from langchain_core.documents import Document

# documents = ""
# qa_pairs = []


# for item in dataset:
#     if(len(item["answers"]["text"])):
#         ans=item["answers"]["text"][0]
#     else:
#         ans=""
#     qa_pairs.append({
#         "question": item["question"],
#         "answer":ans
#     })
#     context = item["context"]

#     # documents.append(
#     #     Document(
#     #         page_content=context
#     #     )
#     # )
#     documents=documents+item["context"]+"\n"

# # print("Total docs:", len(documents))

Total docs: 77958


In [ ]:
# #KG implementation
# from dataclasses import dataclass
# from typing import List, Dict

# @dataclass
# class Triple:
#     subj: str
#     rel: str
#     obj: str

# class SimpleKG:
#     def __init__(self):
#         self.triples: List[Triple] = []

#     def add_triple(self, subj: str, rel: str, obj: str):
#         self.triples.append(Triple(subj, rel, obj))

#     def find_triples(self, entity: str) -> List[Triple]:
#         # return all triples where entity is subject or object
#         return [t for t in self.triples if t.subj == entity or t.obj == entity]


# KG = SimpleKG()
# import spacy

# nlp = spacy.load("en_core_web_sm")

# def extract_triples_spacy(text):
#     doc = nlp(text)
#     triples = []
#     for token in doc:
#         if token.dep_ in ("ROOT", "relcl"):  # verbs or relations
#             subj = [w.text for w in token.lefts if w.dep_ in ("nsubj", "nsubjpass")]
#             obj = [w.text for w in token.rights if w.dep_ in ("dobj", "pobj", "attr")]
#             if subj and obj:
#                 triples.append((" ".join(subj), token.lemma_, " ".join(obj)))
#     return triples




# #link entities in query to KG entities
# import spacy
# nlp = spacy.load("en_core_web_sm")
# def extract_entity_mentions(text):
#     doc = nlp(text)
#     return [ent.text for ent in doc.ents] or [chunk.text for chunk in doc.noun_chunks]

# def link_entities(query, kg_entities):
#     # simple substring + optional embedding similarity
#     mentions = extract_entity_mentions(query)  
#     entity_map = {}
#     for m in mentions:
#         entity_map[m] = [e for e in kg_entities if m.lower() in e.lower()]
#     return entity_map

# def retrieve_kg_context(query, KG: SimpleKG):
#     kg_entities = list(set([t.subj for t in KG.triples] + [t.obj for t in KG.triples]))
#     entity_map = link_entities(query, kg_entities)
#     triples_text = []
#     for ents in entity_map.values():
#         for ent in ents:
#             for t in KG.find_triples(ent):
#                 triples_text.append(f"{t.subj} {t.rel} {t.obj}")
#     return "\n".join(triples_text)



In [ ]:

#combine kg retrieval and context retrieval
# def get_combined_context(query, retriever, KG):
#     # 1. Retrieve text from FAISS DB
#     text_docs = retriever.invoke(query)
#     text_context = "\n\n".join([d.page_content for d in text_docs])
#     #print(f"context:{text_context}")
#     if(len(text_docs)==0):
#         return None
    
#     # 2. Retrieve KG triples
#     kg_context = retrieve_kg_context(query, KG)

#     # 3. Combine for final LLM input
#     combined_context = f"KG Facts:\n{kg_context}\n\nTextual Context:\n{text_context}"
#     return combined_context
# ----------------------------------------------------------------------------------------------------
# def get_combined_context(query, retriever, KG):
#     # 1. Text Retrieval & Deduplication
#     text_docs = retriever.invoke(query)[:3]
    
    
#     unique_texts = list(dict.fromkeys([d.page_content.strip() for d in text_docs]))
#     clean_text = "\n\n".join(unique_texts)

#     # 2. KG Retrieval & Deduplication
#     kg_raw = retrieve_kg_context(query, KG)[:100] # Assuming this returns a string or list
    
    
#     # If it's a string with newlines:
#     if isinstance(kg_raw, str):
#         kg_lines = [line.strip() for line in kg_raw.split('\n') if line.strip()]
#         unique_kg = list(dict.fromkeys(kg_lines))
#         clean_kg = "\n".join(unique_kg)
#     # If it's already a list:
#     else:
#         unique_kg = list(dict.fromkeys([str(fact).strip() for fact in kg_raw]))
#         clean_kg = "\n".join(unique_kg)

#     return f"KG Facts:\n{clean_kg}\n\nTextual Context:\n{clean_text}"

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,     
    chunk_overlap=100,   
    separators=["\n\n", "\n", " ", ""]
)

questions=[]
ground_truths=[]
doc=""
q=0
for row in dataset:
    questions.append(row["question"])
    doc=doc+row["context"]+"\n"
    if(len(row["answers"]["text"])):
        ground_truths.append(row["answers"]["text"][0])
    else:
        ground_truths.append("")
    
chunks= text_splitter.split_text(doc)
print("Total chunks:", len(chunks))





Total chunks: 124


In [7]:
print(chunks[100])

A few years after the First Crusade, in 1107, the Normans under the command of Bohemond, Robert's son, landed in Valona and besieged Dyrrachium using the most sophisticated military equipment of the time, but to no avail. Meanwhile, they occupied Petrela, the citadel of Mili at the banks of the river Deabolis, Gllavenica (Ballsh), Kanina and Jericho. This time, the Albanians sided with the Normans, dissatisfied by the heavy taxes the Byzantines had imposed upon them. With their help, the Normans secured the Arbanon passes and opened their way to Dibra. The lack of supplies, disease and Byzantine resistance forced Bohemond to retreat from his campaign and sign a peace treaty with the Byzantines in the city of Deabolis.


In [8]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2",
                                   model_kwargs={"device": "cuda:0"},
                                   encode_kwargs={"normalize_embeddings": True}
                                   )

/tmp/ipykernel_2095588/2493071643.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2",
/mnt/nas/shuvranshu/.conda/envs/ragenv/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/mnt/nas/shuvranshu/.conda/envs/ragenv/lib/python3.10/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before buildin

In [9]:
#FAISS cosine similarity index
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore

dimension = 784


index = faiss.IndexFlatIP(dimension)

In [10]:

vectorstore = FAISS.from_texts(chunks, embeddings)
vectorstore.save_local("faiss_index")
print(len(chunks))
print(vectorstore.index.ntotal)

124
124


In [61]:

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 20}
    )

In [12]:
llm = HuggingFacePipeline.from_model_id(
    # model_id="/mnt/nas/shuvranshu/huggingface_cache/models--meta-llama--Llama-3.1-8B/snapshots/d04e592bb4f6aa9cfee91e2e20afa771667e1d4b", 
    # model_id="meta-llama/Llama-3.1-8B",
    model_id="/mnt/nas/shuvranshu/finetune/merged_model",
    task="text-generation",
    pipeline_kwargs={
        "temperature": 0.1,
        "max_new_tokens": 256,
        "max_length": None,
        "do_sample": False,
    },
    device_map={"":0}
)


Loading weights: 100%|██████████| 291/291 [00:04<00:00, 61.71it/s]
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_length', 'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [13]:

# You are a factual assistant whose goal is to produce answers strictly supported by the provided context.



# Instructions:
# 1. Use ONLY the information present in the provided context.
# 2. Do NOT introduce external knowledge.
# 3. If the answer cannot be found in the context, respond with No answer.
# 4. Ensure the answer directly addresses the question.
# 5. Answer in maximum 3 to 5 words
# 6. If the answer is implied, infer it.
# 7. If the answer is Rollo then give no answer at all.


# Context:
# {context}

# Question:
# {question}

# Refined Answer:
actor_prompt = PromptTemplate(
    input_variables=["context", "question","feedback_section"],
    template="""
You are a factual assistant whose goal is to produce answers strictly supported by the provided context.



Instructions:
1. Use ONLY the information present in the provided context.
2. Do NOT introduce external knowledge.
3. If the answer cannot be found in the context, don't give any answer.
4. Ensure the answer directly addresses the question.
5. Answer in maximum 3 to 5 words
6. If the answer is implied, infer it.
7. If the answer is Rollo then give no answer at all.


Context:
{context}

Question:
{question}

Refined Answer:
"""
)

def actor(question, context, feedback_section):
    actor_chain = actor_prompt | llm
    # context = get_combined_context(question,retriever, KG)
    response = actor_chain.invoke({
            "context":context,
            "feedback_section": feedback_section,
            "question":question
            }
            )
    answer=response.split('Answer:')[-1].strip()
    return answer


In [ ]:
# prompt = PromptTemplate(
#     input_variables=["context", "question"],
#     template="""
# You are a factual assistant. Use the following context to answer the question.
# Do NOT add information that is not supported by the context.
# if you don't find the answer in the context then simply say NO.

# Context:
# {context}

# Question: {question}
# Answer:
# """
# )

In [ ]:
# question="summarize this document"
# context = get_combined_context(question,retriever, KG)
# chain = prompt | llm
# response = chain.invoke({
#         "context":context,
#         "question":question,
#         }
#         )
# answer=response.split('Answer:')[-1].strip()
# print(f"answer:{answer}")    


/mnt/nas/shuvranshu/.conda/envs/ragenv/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/mnt/nas/shuvranshu/.conda/envs/ragenv/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


answer:The descendants of Rollo's Vikings and their Frankish wives would replace the Norse religion and Old Norse language with Catholicism (Christianity) and the Gallo-Romance language of the local people, blending their maternal Frankish heritage with Old Norse traditions and customs to synthesize a unique "Norman" culture in the north of France. The Norman language was forged by the adoption of the indigenous langue d'oïl branch of Romance by a Norse-speaking ruling class, and it developed into the regional language that survives today.


In [ ]:
# #evaluator
# import json
# from langchain_community.chat_models import ChatOllama


In [ ]:
# evaluator_llm= ChatOllama(model="qwen2.5:7b",
#                           format="json",
#                           temperature=0)
# eval_prompt = ChatPromptTemplate.from_messages([
#     ("system", """You are a RAG Quality Auditor. You must evaluate ONLY two things:
#     1. CONTEXT RELEVANCE: Does the retrieved context contain the information needed to answer the user's query?
#     2. FAITHFULNESS: Is the provided answer derived ONLY from the retrieved context?
     
#     Output ONLY a JSON object with EXACTLY this structure:
#     {{
#         "context_relevance": {{
#             "score": float (0.0 to 1.0),
#             "reason": "string"
#         }},
#         "faithfulness": {{
#             "score": float (0.0 to 1.0),
#             "reason": "string"
#         }},
#         "final_status": "PASS" or "FAIL",
#         "action_item": "What should the system do next?(e.g., 'Re-retrieve context' or 'Refine answer').
#         if context_relevance score<0.8 then 'Re-retrieve context',if faithfullness score<0.8 then 'Refine answer'.
#     }}
    
#     Threshold for PASS: Both scores must be >= 0.8.
#     """),
#     ("user", "USER QUERY: {query}\n\nRETRIEVED CONTEXT: {context}\n\nGENERATED ANSWER: {answer}")
# ])

In [ ]:
# def evaluate_response(question,context, answer):
#     chain = eval_prompt | evaluator_llm
#     # response = chain.invoke({"context": context, "answer": answer})
    
    
#     try:
#         response = chain.invoke({
#             "query": question,
#             "context": context,
#             "answer": answer
#         })
#         return json.loads(response.content)
#     except Exception as e:
#         return {
#             "final_status": "FAIL",
#             "action_item": f"Error in evaluation: {str(e)}"
#         }

In [ ]:

# try:
#     response = evaluator_llm.invoke("Hello!")
#     print("Ollama is awake and responding!")
# except Exception as e:
#     print(f"Connection failed. Is the server running? Error: {e}")

Ollama is awake and responding!


In [ ]:
# query_refiner_llm = ChatOllama(model="qwen2.5:7b", temperature=0)

# def generate_refined_query(original_query, critique):
#     prompt = f"""
#     The initial search for '{original_query}' failed.
#     Critique: {critique}
#     based on this critique, write a better search query which contains the missing terms or concepts that the original query lacked,
#     which led to the failure.
#     Don't change the meaning of the question.
#     Output ONLY the new search string.
#     """
#     response = query_refiner_llm.invoke(prompt)
#     return response.content.strip()

In [ ]:
# def run_reflexion_loop(original_question,max_iterations=3):
    
#     search_query = original_question
#     current_context = get_combined_context(search_query, retriever, KG)
#     feedback_section = " " 
    
#     for i in range(max_iterations):
#         # print(f"\n--- Iteration {i+1} ---")
        
#         answer = actor(original_question, current_context, feedback_section)
        
#         evaluation = evaluate_response(original_question, current_context, answer)
#         # print(json.dumps(evaluation, indent=2))
#         # print(current_context)
#         # print(f"answer:{answer}")

#         if evaluation.get('final_status') == "PASS":
#             return answer,current_context
        
#         action = evaluation.get('action_item', 'Refine answer')
        
#         if action == "Re-retrieve context":
#             search_query = generate_refined_query(original_question, evaluation['context_relevance']['reason'])
#             # print(f"new search query:{search_query}")
#             current_context = get_combined_context(search_query, retriever, KG)
#         else:
#             feedback_section=evaluation['faithfulness']['reason']
#         # feedback_section = f"""
#         # ### FEEDBACK FROM PREVIOUS ATTEMPT ###
#         # - Context Relevance Score: {evaluation['context_relevance']['score']}
#         # - Context Critique: {evaluation['context_relevance']['reason']}
        
#         # - Faithfulness Score: {evaluation['faithfulness']['score']}
#         # - Faithfulness Critique: {evaluation['faithfulness']['reason']}
        
#         # - Required Action: {evaluation['action_item']}
#         # ######################################
#         # """
        
#         # print(f"Critique sent to Actor: {evaluation['action_item']}")
        
#     # print(" Max iterations reached.")
#     return answer,current_context

In [14]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
import json,re

In [57]:
# score_prompt = score_prompt = ChatPromptTemplate.from_messages([
#     ("system", """You are a relevance scoring engine. Given a user question and a retrieved document chunk, output a single relevance score.

# Scoring Rubric:
# - 0.0–0.2: Completely irrelevant.
# - 0.3–0.4: Weakly relevant.
# - 0.5–0.6: Moderately relevant.
# - 0.7–0.8: Highly relevant.
# - 0.9–1.0: Perfectly relevant.

# Respond ONLY with a valid JSON object. No explanation, no preamble.
# {{"score": <float>, "reason": "<one sentence>"}}"""),
#     ("human", "Question: {question}\nDocument: {context}")
# ])
score_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a relevance scoring engine. 
Given a user question and a document, score how useful the document is for answering the question.

IMPORTANT: Even if the document does not explicitly state the answer in one sentence, 
if the answer CAN BE INFERRED from the document, give a HIGH score.

Scoring Rubric:
- 0.0–0.2: Document has nothing to do with the question.
- 0.3–0.4: Document mentions the topic but cannot help answer the question.
- 0.5–0.6: Document contains some relevant information.
- 0.7–0.8: Document contains enough information to answer the question.
- 0.9–1.0: Document directly and clearly answers the question.

Example:
Question: In what country is Normandy located?
Document: "...the French lands... one of the largest fiefs in France..."
Output: {{"score": 0.9, "reason": "The document mentions France multiple times in relation to Normandy."}}

Respond ONLY with a valid JSON object. No other text.
{{"score": <float>, "reason": "<one sentence>"}}"""),
    ("human", "Question: {question}\nDocument: {context}")
])

def score(question, context):
    score_chain = score_prompt | llm 
    raw_output = score_chain.invoke({
        "context": context,   # extract text from Document object
        "question": question
    })
    text = raw_output.content if hasattr(raw_output, 'content') else str(raw_output)
    matches = re.findall(r'"score"\s*:\s*([\d.]+)\s*,\s*"reason"\s*:\s*"((?:[^"\\]|\\.)*)"', text)

    if matches:
        score_val = float(matches[-1][0]) 
        reason = matches[-1][1]
        # print(f"Score: {score_val}")
        # print(f"Reason: {reason}")
        return score_val, reason
    else:
        print("No match found")
    

In [16]:
import heapq


In [64]:
def rank_contexts(question,top_k=3):
    contexts = retriever.invoke(question)
    unique_contexts = list(dict.fromkeys([d.page_content.strip() for d in contexts]))
    scored_contexts = []
    for context in unique_contexts:
        score_val, reason = score(question, context)
        if score_val is not None:
            scored_contexts.append((score_val, reason, context))
    top_contexts = sorted(scored_contexts, key=lambda x: x[0], reverse=True)[:top_k]
    final_context = "\n\n".join([ctx for _, _, ctx in top_contexts])
    return final_context

In [65]:
results = []
rag_answers=[]
retrieved_contexts=[]
q=0

for question in questions:
    context = rank_contexts(question)
    answer = actor(question, context, feedback_section=" ")
    retrieved_contexts.append(context)
    ans=answer.split('Answer:')[-1].strip()
    rag_answers.append(ans)
    print(f"qa {q}:{ans}")
    q+=1

qa 0:France
qa 1:first half of the 10th century
qa 2:Denmark, Iceland and Norway
qa 3:Rollo
qa 4:10th and 11th centuries
qa 5:The Normans
qa 6:No answer
qa 7:no answer
qa 8:first half of the 10th century
qa 9:Duke William II of Normandy
qa 10:Duke William II of Normandy
qa 11:Catholic orthodoxy
qa 12:no answer
qa 13:No answer
qa 14:No answer
qa 15:Duke William II of Normandy
qa 16:no answer
qa 17:Norseman, Viking
qa 18:9th century
qa 19:The Normans
qa 20:9th century
qa 21:911
qa 22:King Charles III of West Francia
qa 23:Epte
qa 24:no answer
qa 25:no answer
qa 26:Rollo
qa 27:Viking incursions
qa 28:Rollo
qa 29:880s
qa 30:Danes, Norwegians, Norse–Gaels, Orkney Vikings, possibly Swedes, and Anglo-Danes from the English Danelaw under Norse control
qa 31:Catholic orthodoxy
qa 32:Upper Normandy
qa 33:Christianity
qa 34:Frankish heritage
qa 35:no answer
qa 36:fighting horsemen
qa 37:No answer
qa 38:feudal doctrines
qa 39:Normans
qa 40:Seljuk Turks
qa 41:Normans
qa 42:the Pechenegs, the Bulgar

In [66]:
#EM
# correct = sum([pred.strip().lower() == gt.strip().lower()
#                for pred, gt in zip(rag_answers, ground_truths)])

# total = len(ground_truths)

# print("Correct:", correct)
# print("Accuracy:", correct / total)
def normalize(text):
    text = text.strip().lower()
    if text == "no answer":
        return ""
    return text

correct = sum([
    normalize(pred) == normalize(gt)
    for pred, gt in zip(rag_answers, ground_truths)
])

total = len(ground_truths)

print("Correct:", correct)
print("Accuracy:", correct / total)

Correct: 40
Accuracy: 0.4


In [68]:
#substring match
# correct = sum([gt.lower() in pred.lower()
#                for pred, gt in zip(rag_answers, ground_truths)])

# print("Accuracy:", correct / len(ground_truths))

def normalize(text):
    text = text.strip().lower()
    if text in ["no answers", "no answer", "no_answer", "n/a"]:
        return ""
    return text

correct = sum([
    normalize(gt) in normalize(pred)
    for pred, gt in zip(rag_answers, ground_truths)
])

print("Accuracy:", correct / len(ground_truths))

Accuracy: 0.87


In [ ]:
# #exact match and F1 score
# import re
# import string

# def normalize_answer(s):
#     def remove_articles(text):
#         return re.sub(r'\b(a|an|the)\b', ' ', text)

#     def white_space_fix(text):
#         return ' '.join(text.split())

#     def remove_punc(text):
#         return ''.join(ch for ch in text if ch not in set(string.punctuation))

#     def lower(text):
#         return text.lower()

#     return white_space_fix(remove_articles(remove_punc(lower(s))))

In [ ]:
# def compute_em(pred, gt):
#     return int(normalize_answer(pred) == normalize_answer(gt))

In [73]:
from collections import Counter

def compute_f1(pred, gt):
    pred_tokens = normalize(pred).split()
    gt_tokens = normalize(gt).split()

    common = Counter(pred_tokens) & Counter(gt_tokens)
    num_common = sum(common.values())

    if num_common == 0:
        return 0.0

    precision = num_common / len(pred_tokens)
    recall = num_common / len(gt_tokens)

    return 2 * precision * recall / (precision + recall)

In [74]:
f1_scores = []

for pred, gt in zip(rag_answers, ground_truths):
    

    if pred is None or pred.strip() == "":
        continue   # skip bad outputs

    # em_scores.append(compute_em(pred, gt))
    f1_scores.append(compute_f1(pred, gt))

# avg_em = sum(em_scores) / len(em_scores)
avg_f1 = sum(f1_scores) / len(f1_scores)

# print("Exact Match (EM):", avg_em)
print("F1 Score:", avg_f1)

F1 Score: 0.2864642857142857


In [75]:

dataset_ragas = []

for q, ctx, ans, ref in zip(questions, retrieved_contexts, rag_answers, ground_truths):
    dataset_ragas.append({
        "user_input": q,
        "retrieved_contexts": ctx if isinstance(ctx, list) else [ctx],
        "response": ans,
        "reference": ref
    })

In [76]:
from ragas import EvaluationDataset
evaluation_dataset = EvaluationDataset.from_list(dataset_ragas)

In [77]:
from langchain_community.chat_models import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from ragas import evaluate
from langchain_community.embeddings import OllamaEmbeddings
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness
# answer_relevancy, context_precision

In [78]:
json_prompt = """
You are a strict JSON output generator used for evaluation in RAG systems.
You must always reply ONLY with a valid JSON object — no explanations, no extra text.

If the question asks for a score, output: {"score": <float between 0 and 1>}
If the question asks for a label (e.g., yes/no), output: {"label": "yes"} or {"label": "no"}
Do not include extra keys, comments, or markdown fences.
"""
langchain_llm= ChatOllama(model="mistral",temperature=0,system=json_prompt)


langchain_embeddings=OllamaEmbeddings(model="nomic-embed-text")

/tmp/ipykernel_2095588/241535895.py:9: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  langchain_llm= ChatOllama(model="mistral",temperature=0,system=json_prompt)
/tmp/ipykernel_2095588/241535895.py:12: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  langchain_embeddings=OllamaEmbeddings(model="nomic-embed-text")


In [79]:
result = evaluate(dataset=evaluation_dataset,
                  batch_size=2,
                #   metrics=[ LLMContextRecall(), Faithfulness(),FactualCorrectness()],
                  metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness()],
                  llm=langchain_llm,embeddings=langchain_embeddings)
print(result)

Evaluating:   9%|▉         | 28/300 [01:47<12:18,  2.71s/it] Exception raised in Job[29]: OutputParserException(Failed to parse StringIO from completion {"claims": ["Duke William II existed.", "William II held the title Duke.", "William II was a member of the Norman dynasty.", "William II ruled over Normandy."]}. Got: 1 validation error for StringIO
text
  Field required [type=missing, input_value={'claims': ['Duke William... ruled over Normandy.']}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/missing
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE )
Evaluating:  11%|█         | 32/300 [02:01<13:36,  3.05s/it]Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt fix_output_format failed to parse output: The outp

{'context_recall': 0.4325, 'faithfulness': 0.6417, 'factual_correctness(mode=f1)': 0.4337}


In [80]:
from sentence_transformers import SentenceTransformer, util
import torch
import numpy as np

qa_model = SentenceTransformer("multi-qa-MiniLM-L6-cos-v1", device="cuda:0")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 976.47it/s]
BertModel LOAD REPORT from: sentence-transformers/multi-qa-MiniLM-L6-cos-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [81]:

scores = []
skipped = 0
for row in evaluation_dataset:
    if not row.reference or row.reference.strip() == "":  # skip empty references
        skipped += 1
        continue
    
    ref_emb = qa_model.encode(row.reference, convert_to_tensor=True)
    ans_emb = qa_model.encode(row.response, convert_to_tensor=True)
    score = util.cos_sim(ref_emb, ans_emb).item()
    scores.append(score)

print(f"Skipped {skipped} samples with no reference")
print(f"Valid samples: {len(scores)}/{len(evaluation_dataset)}")
print(f"answer_similarity (generated vs reference): {np.mean(scores):.4f}")

# def normalize(text):
#     if not text:
#         return ""
#     text = text.strip().lower()
#     if text in ["no answers", "no answer", "no_answer", "n/a"]:
#         return ""
#     return text

# scores = []
# skipped = 0

# for row in evaluation_dataset:
#     # normalize both reference and response
#     reference = normalize(row.reference)
#     response = normalize(row.response)

#     # skip if reference is empty
#     if reference == "":
#         skipped += 1
#         continue

#     # handle case where response becomes empty
#     if response == "":
#         score = 0.0   # no answer → worst similarity
#     else:
#         ref_emb = qa_model.encode(reference, convert_to_tensor=True)
#         ans_emb = qa_model.encode(response, convert_to_tensor=True)
#         score = util.cos_sim(ref_emb, ans_emb).item()

#     scores.append(score)

# print(f"Skipped {skipped} samples with no reference")
# print(f"Valid samples: {len(scores)}/{len(evaluation_dataset)}")
# print(f"answer_similarity (generated vs reference): {np.mean(scores):.4f}")

Skipped 55 samples with no reference
Valid samples: 45/100
answer_similarity (generated vs reference): 0.7872
